# SigLIP2 Visual Substitute Analysis

This notebook generates the top 10 visually similar products for every product in the held-out SigLIP2 furniture catalog and evaluates whether those neighbors look like plausible substitutes according to the product taxonomy.

The core question is: **when a product is a sofa, chair, table, bed, or another furniture type, does visual retrieval return the same or a closely related furniture type?**

## 1. Prerequisites and scope

Select the `RecSystem — SigLIP2 (Python 3.11)` kernel.

This analysis uses the 500 unseen catalog products and image embeddings produced by the winning `multichunk64` arm in `siglip2_furniture_text_strategy_benchmark.ipynb`. It performs exact nearest-neighbor search over all 500 products. The same evaluation code can be reused with a larger embedding catalog; a 20,000-product production catalog should use a nearest-neighbor index such as FAISS rather than materializing a full pairwise similarity matrix.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image, ImageOps

SEED = 42
TOP_K = 10
MIN_CATEGORY_PRODUCTS = 5

def find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "notebooks").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate the RecSystem project root.")

ROOT = find_project_root()
SOURCE_DIR = ROOT / "models" / "siglip2_furniture_text_strategy_benchmark"
CATALOG_PATH = SOURCE_DIR / "embedding_catalog.csv"
IMAGE_EMBEDDINGS_PATH = SOURCE_DIR / "multichunk64_image_embeddings.npy"
OUTPUT_DIR = ROOT / "models" / "siglip2_furniture_visual_substitute_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print({"project_root": str(ROOT), "output_directory": str(OUTPUT_DIR)})

In [ ]:
missing = [path for path in [CATALOG_PATH, IMAGE_EMBEDDINGS_PATH] if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing benchmark artifacts:\n"
        + "\n".join(f"- {path}" for path in missing)
        + "\nRun siglip2_furniture_text_strategy_benchmark.ipynb first."
    )

catalog = pd.read_csv(CATALOG_PATH, dtype={"asin": "string"}).reset_index(drop=True)
image_embeddings = np.load(IMAGE_EMBEDDINGS_PATH).astype(np.float32)

required_columns = {
    "asin", "description_original", "furniture_subcategory",
    "category_path", "leaf_category", "image_path",
}
assert required_columns.issubset(catalog.columns)
assert len(catalog) == len(image_embeddings)
assert catalog["asin"].is_unique
assert np.isfinite(image_embeddings).all()
assert np.allclose(np.linalg.norm(image_embeddings, axis=1), 1.0, atol=1e-4)

print({
    "catalog_products": len(catalog),
    "embedding_shape": image_embeddings.shape,
    "leaf_categories": catalog["leaf_category"].nunique(),
    "room_categories": catalog["furniture_subcategory"].nunique(),
})

## 2. Define substitute relevance from taxonomy

We evaluate each retrieved neighbor at three cumulative strengths:

- **Strict substitute:** the same exact leaf category, such as `Sofas & Couches → Sofas & Couches`.
- **Relaxed family substitute:** the same leaf **or** the same immediate taxonomy parent, such as `Coffee Tables → End Tables`, or `Sofas & Couches → Chairs`.
- **Contextual match:** a relaxed family match **or** the same room-level category. This is the weakest test; a sofa and media cabinet can both be living-room products without being substitutes.

For graded ranking quality, relevance is 3 for the same leaf, 2 for a sibling under the same immediate parent, 1 for the same room, and 0 otherwise. This supports NDCG@10, which rewards highly relevant taxonomy matches appearing earlier.

These are taxonomy-proxy metrics. They do not prove that two products match in dimensions, price, material, style, or customer intent.

In [ ]:
def taxonomy_parent(category_path):
    parts = [part.strip() for part in str(category_path).split(">") if part.strip()]
    return " > ".join(parts[:-1]) if len(parts) > 1 else "Unspecified"

catalog["taxonomy_parent"] = catalog["category_path"].map(taxonomy_parent)
catalog["leaf_category"] = catalog["leaf_category"].fillna("Unspecified")
catalog["furniture_subcategory"] = catalog["furniture_subcategory"].fillna("Unspecified")

taxonomy_preview = catalog[[
    "furniture_subcategory", "taxonomy_parent", "leaf_category", "category_path"
]].drop_duplicates().head(20)
display(taxonomy_preview)

## 3. Generate exact top-10 visual neighbors for every product

The image embeddings are unit-normalized, so their matrix product is cosine similarity. Self-similarity is removed before ranking. No taxonomy is used during retrieval; taxonomy is only used afterward for evaluation.

In [ ]:
similarities = image_embeddings @ image_embeddings.T
np.fill_diagonal(similarities, -np.inf)
neighbor_indices = np.argsort(-similarities, axis=1, kind="stable")[:, :TOP_K]
neighbor_scores = np.take_along_axis(similarities, neighbor_indices, axis=1)

query_indices = np.repeat(np.arange(len(catalog)), TOP_K)
candidate_indices = neighbor_indices.reshape(-1)
ranks = np.tile(np.arange(1, TOP_K + 1), len(catalog))

leaf = catalog["leaf_category"].astype(str).to_numpy()
parent = catalog["taxonomy_parent"].astype(str).to_numpy()
room = catalog["furniture_subcategory"].astype(str).to_numpy()

same_leaf_matrix = leaf[neighbor_indices] == leaf[:, None]
same_parent_matrix = parent[neighbor_indices] == parent[:, None]
same_room_matrix = room[neighbor_indices] == room[:, None]
family_match_matrix = same_leaf_matrix | same_parent_matrix
context_match_matrix = family_match_matrix | same_room_matrix
relevance_matrix = np.where(
    same_leaf_matrix, 3, np.where(same_parent_matrix, 2, np.where(same_room_matrix, 1, 0))
)

neighbor_pairs = pd.DataFrame({
    "query_index": query_indices,
    "rank": ranks,
    "candidate_index": candidate_indices,
    "visual_similarity": neighbor_scores.reshape(-1),
    "query_asin": catalog.loc[query_indices, "asin"].to_numpy(),
    "candidate_asin": catalog.loc[candidate_indices, "asin"].to_numpy(),
    "query_leaf_category": leaf[query_indices],
    "candidate_leaf_category": leaf[candidate_indices],
    "query_parent_category": parent[query_indices],
    "candidate_parent_category": parent[candidate_indices],
    "query_room_category": room[query_indices],
    "candidate_room_category": room[candidate_indices],
    "same_leaf": same_leaf_matrix.reshape(-1),
    "same_immediate_parent": same_parent_matrix.reshape(-1),
    "relaxed_family_match": family_match_matrix.reshape(-1),
    "same_room": same_room_matrix.reshape(-1),
    "contextual_match": context_match_matrix.reshape(-1),
    "taxonomy_relevance_grade": relevance_matrix.reshape(-1),
    "query_description": catalog.loc[query_indices, "description_original"].to_numpy(),
    "candidate_description": catalog.loc[candidate_indices, "description_original"].to_numpy(),
    "query_image_path": catalog.loc[query_indices, "image_path"].to_numpy(),
    "candidate_image_path": catalog.loc[candidate_indices, "image_path"].to_numpy(),
})

assert len(neighbor_pairs) == len(catalog) * TOP_K
assert not neighbor_pairs["query_asin"].eq(neighbor_pairs["candidate_asin"]).any()
display(neighbor_pairs.head(10))

## 4. Calculate substitute-quality metrics

We report precision at 1, 5, and 10 for every taxonomy strength, strict hit rate@10, strict mean reciprocal rank, and graded NDCG@10. We also calculate the expected match rate from uniformly random non-self retrieval so category imbalance does not make a raw precision number look better than it is.

In [ ]:
discounts = 1.0 / np.log2(np.arange(2, TOP_K + 2))
actual_dcg = ((2 ** relevance_matrix - 1) * discounts).sum(axis=1)

all_same_leaf = leaf[:, None] == leaf[None, :]
all_same_parent = parent[:, None] == parent[None, :]
all_same_room = room[:, None] == room[None, :]
all_family_match = all_same_leaf | all_same_parent
all_context_match = all_family_match | all_same_room
all_relevance = np.where(
    all_same_leaf, 3, np.where(all_same_parent, 2, np.where(all_same_room, 1, 0))
).astype(float)
np.fill_diagonal(all_relevance, -1)
ideal_relevance = np.sort(all_relevance, axis=1)[:, ::-1][:, :TOP_K]
ideal_dcg = ((2 ** ideal_relevance - 1) * discounts).sum(axis=1)
ndcg_at_10 = np.divide(actual_dcg, ideal_dcg, out=np.zeros_like(actual_dcg), where=ideal_dcg > 0)

strict_first_rank = np.array([
    np.flatnonzero(matches)[0] + 1 if matches.any() else 0 for matches in same_leaf_matrix
])
strict_mrr = np.where(strict_first_rank > 0, 1.0 / np.maximum(strict_first_rank, 1), 0.0)

query_metrics = catalog[["asin", "furniture_subcategory", "taxonomy_parent", "leaf_category"]].copy()
for k in (1, 5, 10):
    query_metrics[f"strict_precision@{k}"] = same_leaf_matrix[:, :k].mean(axis=1)
    query_metrics[f"family_precision@{k}"] = family_match_matrix[:, :k].mean(axis=1)
    query_metrics[f"context_precision@{k}"] = context_match_matrix[:, :k].mean(axis=1)
query_metrics["strict_hit@10"] = same_leaf_matrix.any(axis=1).astype(float)
query_metrics["strict_mrr@10"] = strict_mrr
query_metrics["graded_ndcg@10"] = ndcg_at_10

n = len(catalog)
random_expectations = {
    "strict": float(((all_same_leaf.sum(axis=1) - 1) / (n - 1)).mean()),
    "family": float(((all_family_match.sum(axis=1) - 1) / (n - 1)).mean()),
    "context": float(((all_context_match.sum(axis=1) - 1) / (n - 1)).mean()),
}

metric_matrices = {
    "strict": same_leaf_matrix,
    "family": family_match_matrix,
    "context": context_match_matrix,
}
summary_rows = []
for level, matrix in metric_matrices.items():
    for k in (1, 5, 10):
        model_value = float(matrix[:, :k].mean())
        random_value = random_expectations[level]
        summary_rows.append({
            "taxonomy_strength": level,
            "k": k,
            "model_precision": model_value,
            "random_expectation": random_value,
            "lift_vs_random": model_value / random_value if random_value else np.nan,
        })
aggregate_metrics = pd.DataFrame(summary_rows)

additional_metrics = pd.DataFrame([{
    "strict_hit@10": query_metrics["strict_hit@10"].mean(),
    "strict_mrr@10": query_metrics["strict_mrr@10"].mean(),
    "graded_ndcg@10": query_metrics["graded_ndcg@10"].mean(),
}])
display(aggregate_metrics.round(4))
display(additional_metrics.round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
level_titles = {
    "strict": "Strict: same leaf",
    "family": "Relaxed: same leaf or parent",
    "context": "Context: family or room",
}
for axis, level in zip(axes, ["strict", "family", "context"]):
    view = aggregate_metrics.loc[aggregate_metrics["taxonomy_strength"].eq(level)]
    bars = axis.bar(view["k"].astype(str), view["model_precision"], color="#31688e")
    axis.axhline(random_expectations[level], color="#b23a48", linestyle="--", label="random expectation")
    axis.bar_label(bars, labels=[f"{value:.1%}" for value in view["model_precision"]], padding=3)
    axis.set_title(level_titles[level])
    axis.set_xlabel("Top K")
    axis.set_ylim(0, 0.82)
    axis.legend(loc="upper right", fontsize=8)
axes[0].set_ylabel("Mean taxonomy precision")
fig.suptitle("SigLIP2 visual-neighbor taxonomy agreement")
plt.tight_layout()
plt.show()

## 5. Diagnose performance by product category

Aggregate performance can hide weak categories. The next table and chart include leaf categories with at least five catalog products. Small categories have few possible exact-leaf neighbors, so support size should always be considered alongside precision.

In [ ]:
category_metrics = (
    query_metrics.groupby("leaf_category")
    .agg(
        products=("asin", "size"),
        strict_precision_at_10=("strict_precision@10", "mean"),
        family_precision_at_10=("family_precision@10", "mean"),
        context_precision_at_10=("context_precision@10", "mean"),
        strict_hit_at_10=("strict_hit@10", "mean"),
        graded_ndcg_at_10=("graded_ndcg@10", "mean"),
    )
    .sort_values("strict_precision_at_10", ascending=False)
)
supported_categories = category_metrics.loc[category_metrics["products"].ge(MIN_CATEGORY_PRODUCTS)].copy()
display(supported_categories.round(4))

plot_data = supported_categories.sort_values("strict_precision_at_10")
positions = np.arange(len(plot_data))
fig, axis = plt.subplots(figsize=(11, max(7, 0.34 * len(plot_data))))
axis.barh(positions + 0.18, plot_data["family_precision_at_10"], height=0.34, label="relaxed family P@10", color="#35b779")
axis.barh(positions - 0.18, plot_data["strict_precision_at_10"], height=0.34, label="strict leaf P@10", color="#31688e")
axis.set_yticks(positions, plot_data.index)
axis.set_xlabel("Mean precision@10")
axis.set_ylabel("Query leaf category")
axis.set_title(f"Substitute quality by category (at least {MIN_CATEGORY_PRODUCTS} products)")
axis.legend()
plt.tight_layout()
plt.show()

## 6. Inspect individual top-10 rankings

Taxonomy metrics are only a proxy, so visual review is essential. Each candidate is labeled with its similarity, taxonomy grade, and leaf category. Grade 3 is an exact-leaf match, grade 2 is an immediate-parent sibling, grade 1 shares only the room, and grade 0 is outside those taxonomy relationships.

In [ ]:
asin_to_index = pd.Series(catalog.index, index=catalog["asin"]).to_dict()
GRADE_LABELS = {3: "exact leaf", 2: "same parent", 1: "same room", 0: "outside"}

def load_product_image(image_path, size=(400, 400)):
    path = Path(image_path)
    if not path.is_absolute():
        path = ROOT / path
    with Image.open(path) as opened:
        image = opened.convert("RGB")
    return ImageOps.contain(image, size)

def visual_neighbors(asin):
    asin = str(asin)
    if asin not in asin_to_index:
        raise KeyError(f"ASIN {asin!r} is not in this catalog.")
    return neighbor_pairs.loc[neighbor_pairs["query_asin"].eq(asin)].sort_values("rank").copy()

def show_visual_substitutes(asin):
    query_index = int(asin_to_index[str(asin)])
    query = catalog.iloc[query_index]
    results = visual_neighbors(asin)
    metrics = query_metrics.iloc[query_index]
    print({
        "query_asin": str(asin),
        "leaf_category": query["leaf_category"],
        "room_category": query["furniture_subcategory"],
        "strict_precision@10": round(float(metrics["strict_precision@10"]), 3),
        "family_precision@10": round(float(metrics["family_precision@10"]), 3),
        "graded_ndcg@10": round(float(metrics["graded_ndcg@10"]), 3),
    })
    print("Query description:", str(query["description_original"])[:500])

    table = results[[
        "rank", "visual_similarity", "candidate_asin",
        "candidate_leaf_category", "taxonomy_relevance_grade", "candidate_description"
    ]].copy()
    table["candidate_description"] = table["candidate_description"].str.slice(0, 180)
    display(table.round({"visual_similarity": 4}))

    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    axes = axes.ravel()
    axes[0].imshow(load_product_image(query["image_path"]))
    axes[0].set_title(f"QUERY\n{query['leaf_category']}\n{query['asin']}", fontweight="bold")
    axes[0].axis("off")
    for axis, (_, row) in zip(axes[1:], results.iterrows()):
        grade = int(row["taxonomy_relevance_grade"])
        axis.imshow(load_product_image(row["candidate_image_path"]))
        axis.set_title(
            f"#{int(row['rank'])}  sim={row['visual_similarity']:.3f}\n"
            f"grade {grade}: {GRADE_LABELS[grade]}\n{row['candidate_leaf_category']}",
            fontsize=9,
        )
        axis.axis("off")
    for axis in axes[len(results) + 1:]:
        axis.axis("off")
    fig.suptitle("Top 10 SigLIP2 visual neighbors", fontsize=15)
    plt.tight_layout()
    plt.show()
    return results


In [ ]:
EXAMPLE_QUERIES = {
    "sofa": "B089Q7ST3D",
    "coffee_table": "B07XB1P8Q6",
    "chair": "B0CBCBJL3R",
}
for label, asin in EXAMPLE_QUERIES.items():
    assert asin in asin_to_index, f"Missing expected {label} example {asin}."
    print(label, asin, "->", catalog.iloc[asin_to_index[asin]]["leaf_category"])

In [ ]:
sofa_neighbors = show_visual_substitutes(EXAMPLE_QUERIES["sofa"])

In [ ]:
coffee_table_neighbors = show_visual_substitutes(EXAMPLE_QUERIES["coffee_table"])

In [ ]:
chair_neighbors = show_visual_substitutes(EXAMPLE_QUERIES["chair"])

## 7. Find strong, median, and weak cases

The following cell selects representative queries by graded NDCG@10, limited to categories with at least five examples. This makes failure analysis easy without cherry-picking only attractive results. Copy any selected ASIN into `show_visual_substitutes(...)` for its image grid.

In [ ]:
leaf_support = catalog["leaf_category"].value_counts()
eligible = query_metrics.loc[
    query_metrics["leaf_category"].map(leaf_support).ge(MIN_CATEGORY_PRODUCTS)
].sort_values("graded_ndcg@10")
median_value = eligible["graded_ndcg@10"].median()
median_index = (eligible["graded_ndcg@10"] - median_value).abs().idxmin()
case_indices = {
    "weak": eligible.index[0],
    "median": median_index,
    "strong": eligible.index[-1],
}
review_cases = pd.DataFrame([
    {"case": label, **query_metrics.loc[index].to_dict()} for label, index in case_indices.items()
]).set_index("case")
display(review_cases[[
    "asin", "leaf_category", "strict_precision@10",
    "family_precision@10", "context_precision@10", "graded_ndcg@10"
]].round(4))

# Example: inspect the weakest automatically selected case.
# weak_neighbors = show_visual_substitutes(review_cases.loc["weak", "asin"])

## 8. Save complete results

The product-level outputs are generated artifacts and remain ignored by Git. They can be joined back to the source catalog by ASIN.

In [ ]:
neighbor_pairs.to_csv(OUTPUT_DIR / "visual_substitute_top10.csv", index=False)
query_metrics.to_csv(OUTPUT_DIR / "visual_substitute_query_metrics.csv", index=False)
aggregate_metrics.to_csv(OUTPUT_DIR / "visual_substitute_aggregate_metrics.csv", index=False)
category_metrics.to_csv(OUTPUT_DIR / "visual_substitute_category_metrics.csv")

experiment_summary = {
    "source_model_strategy": "multichunk64",
    "catalog_products": len(catalog),
    "neighbors_per_product": TOP_K,
    "strict_precision@1": float(same_leaf_matrix[:, :1].mean()),
    "strict_precision@5": float(same_leaf_matrix[:, :5].mean()),
    "strict_precision@10": float(same_leaf_matrix.mean()),
    "family_precision@10": float(family_match_matrix.mean()),
    "context_precision@10": float(context_match_matrix.mean()),
    "strict_hit@10": float(query_metrics["strict_hit@10"].mean()),
    "strict_mrr@10": float(query_metrics["strict_mrr@10"].mean()),
    "graded_ndcg@10": float(query_metrics["graded_ndcg@10"].mean()),
    "random_expectations": random_expectations,
}
with (OUTPUT_DIR / "experiment_summary.json").open("w") as output:
    json.dump(experiment_summary, output, indent=2)

print(json.dumps(experiment_summary, indent=2))
print("Saved:", sorted(path.name for path in OUTPUT_DIR.iterdir()))

## 9. Interpretation and next step

A useful model should substantially outperform the random taxonomy-match expectation, perform reasonably across many supported categories, and produce visually convincing examples—not merely a high aggregate number driven by large categories.

The strict metric is conservative: visually interchangeable products can have different leaf labels. The relaxed metric is permissive: taxonomy siblings can still represent complements instead of substitutes. For that reason, the best next evaluation is a human-labeled sample of query-candidate pairs using a rubric such as:

- 2: plausible direct substitute
- 1: related but not a direct substitute
- 0: irrelevant

That sample can measure precision@10 and NDCG@10 against human judgments, and can reveal whether the taxonomy proxy is too strict or too generous for particular product families.